# 05 · Decision tree


## Load and split data

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

df = pd.read_csv(root / "data/raw/online_shoppers_intention.csv").drop_duplicates()
X = df.drop(columns="Revenue")
X["Weekend"] = X["Weekend"].astype(int)
y = df["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Preprocessing

Keep the same features, scaling and encoding as notebook 04. Trees do not require scaling, but retaining the pipeline keeps preparation consistent. Preprocessing is fitted separately within each training fold.

In [2]:
numeric_features = [
    "Administrative", "Administrative_Duration", "Informational",
    "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
]
categorical_features = [
    "Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType",
]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("weekend", "passthrough", ["Weekend"]),
])

## Training cross-validation

Use the same five stratified folds and purchase-class metrics as earlier notebooks. Tables show fold mean ± standard deviation.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_folds = list(cv.split(X_train, y_train))

scoring = {
    "f1": make_scorer(f1_score, pos_label=True, zero_division=0),
    "precision": make_scorer(precision_score, pos_label=True, zero_division=0),
    "recall": make_scorer(recall_score, pos_label=True, zero_division=0),
    "accuracy": "accuracy",
}


## Candidate settings

Try tree depths 3, 5 and 8, each with default and balanced class weights: six configurations in total. Set `min_samples_leaf=20` to avoid very small leaves and constrain depth to limit overfitting. Fix `random_state=42`; leave other tree settings at their defaults. No threshold tuning is performed.

In [4]:
candidates = []
for depth in [3, 5, 8]:
    for weight in [None, "balanced"]:
        candidates.append(("Decision tree", DecisionTreeClassifier(
            max_depth=depth, min_samples_leaf=20,
            class_weight=weight, random_state=42,
        )))

## Tree results

Select the configuration with the highest mean purchase-class F1.

In [5]:
rows = []
for family, estimator in candidates:
    model = Pipeline([
        ("preprocessing", preprocessor),
        ("model", estimator),
    ])
    scores = cross_validate(
        model, X_train, y_train, cv=cv_folds, scoring=scoring,
        error_score="raise",
    )
    params = estimator.get_params()
    row = {
        "model": family,
        "class_weight": str(params.get("class_weight")),
        "max_depth": params.get("max_depth"),
        "settings": "min_samples_leaf=20, random_state=42",
    }
    for metric in scoring:
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()
    rows.append(row)
results = pd.DataFrame(rows)
best_index = results.sort_values(
    ["f1_mean", "max_depth"], ascending=[False, True], kind="stable"
).index[0]
results["selected"] = results.index == best_index

comparison = results[["model", "class_weight", "max_depth", "settings", "selected"]].copy()
comparison["max_depth"] = comparison["max_depth"].astype("Int64").astype("string").fillna("-")
for metric in scoring:
    comparison[metric] = [
        f"{mean:.2%} ± {std:.2%}"
        for mean, std in zip(results[f"{metric}_mean"], results[f"{metric}_std"])
    ]
display(comparison)
print("Current best configuration:")
print(comparison.loc[best_index].to_string())

assert len(results) == 6
assert results["selected"].sum() == 1

,model,class_weight,max_depth,settings,selected,f1,precision,recall,accuracy
0,Decision tree,None,3,"min_samples_leaf=20, random_state=42",False,61.65% ± 5.77%,69.26% ± 4.04%,55.83% ± 7.30%,89.25% ± 1.19%
1,Decision tree,balanced,3,"min_samples_leaf=20, random_state=42",False,63.06% ± 1.04%,49.75% ± 1.40%,86.17% ± 1.84%,84.21% ± 0.77%
2,Decision tree,None,5,"min_samples_leaf=20, random_state=42",True,64.43% ± 3.77%,72.63% ± 2.70%,58.06% ± 5.14%,90.02% ± 0.82%
3,Decision tree,balanced,5,"min_samples_leaf=20, random_state=42",False,62.58% ± 1.07%,49.80% ± 1.50%,84.27% ± 1.87%,84.24% ± 0.75%
4,Decision tree,None,8,"min_samples_leaf=20, random_state=42",False,63.26% ± 1.82%,68.87% ± 2.02%,58.52% ± 1.89%,89.38% ± 0.54%
5,Decision tree,balanced,8,"min_samples_leaf=20, random_state=42",False,61.88% ± 0.94%,49.45% ± 1.50%,82.83% ± 2.94%,84.04% ± 0.81%


Current best configuration:
model                                  Decision tree
class_weight                                    None
max_depth                                          5
settings        min_samples_leaf=20, random_state=42
selected                                        True
f1                                    64.43% ± 3.77%
precision                             72.63% ± 2.70%
recall                                58.06% ± 5.14%
accuracy                              90.02% ± 0.82%


## Findings

The selected tree uses `max_depth=5`, `min_samples_leaf=20` and default class weights. Mean purchase F1 is 64.43% (standard deviation 3.77%), precision is 72.63% and recall is 58.06%.

Balanced weights increase recall but reduce precision, giving lower F1 than the selected tree. Increasing depth to 8 does not improve mean F1.